#BTOL: Thuật toán AKT vào trong 8-puzzle

In [9]:
import copy
from heapq import heappush, heappop

#Giả sử 8-puzzle (n=3)
n = 3

#4 vị trí di chuyển lên, xuống, trái, phải
rows = [1,0,-1,0]
cols = [0,-1,0,1]

class priorityQueue:
  def __init__(self):
    self.heap = []

  def push(self, key):
    heappush(self.heap, key)

  def pop(self):
    return heappop(self.heap)

  def empty(self):
    if not self.heap:
      return True
    else:
      return False

class nodes:
  def __init__(self, parent, mats, empty_tile_pos, costs, levels):
    self.parent = parent
    self.mats = mats
    self.empty_tile_pos = empty_tile_pos
    self.costs = costs
    self.levels = levels

  def __lt__(self, nxt):
    return self.costs < nxt.costs

#Tính toán chi phí (tức là tính toán những ô sai so với kết quả muốn trả về)
  def calculateCosts(mats, final) -> int:
    count = 0
    for i in range(n):
      for j in range(n):
        # We only count misplaced tiles that are not the empty tile (0)
        if mats[i][j] != 0 and mats[i][j] != final[i][j]:
          count += 1
    return count

  def newNodes(mats, empty_tile_posi, new_empty_tile_posi, level, parent, final):
    new_mats = copy.deepcopy(mats)
    x1 = empty_tile_posi[0] #Gán x1 = ô trống
    x2 = new_empty_tile_posi[0] #Gán x2 = vị trí ô trống khi di chuyển
    y1 = empty_tile_posi[1]
    y2 = new_empty_tile_posi[1]

    # Swap tiles
    new_mats[x1][y1], new_mats[x2][y2] = new_mats[x2][y2], new_mats[x1][y1]

    # Calculate costs for the new state
    costs = nodes.calculateCosts(new_mats, final)

    # Create and return a new node instance
    new_node = nodes(parent, new_mats, new_empty_tile_posi, costs, level)
    return new_node


  def printMatrix(mats):
    for i in range(n):
      for j in range(n):
        print("%d " % (mats[i][j]), end = " ")
      print()


  def isSafe(x, y):
    return x >=0 and x<n and y >= 0 and y<n


  def printPath(root):
    if root==None:
      return
    nodes.printPath(root.parent)
    nodes.printMatrix(root.mats)
    print()

def solve(initial, empty_tile_posi, final):
  pq = priorityQueue()

  # Initial cost calculation for the root node
  costs = nodes.calculateCosts(initial, final)
  root = nodes(None, initial, empty_tile_posi, costs, 0)

  pq.push(root)
  while not pq.empty():
    minimum = pq.pop()

    # Check if goal state is reached (cost is 0)
    if minimum.costs == 0:
      nodes.printPath(minimum)
      return

    # Explore neighbors
    for i in range(len(rows)): # Iterate through possible moves
      new_tile_posi=[
          minimum.empty_tile_pos[0] + rows[i],
          minimum.empty_tile_pos[1] + cols[i],
      ]

      # If the new position is safe, create a child node
      if nodes.isSafe(new_tile_posi[0], new_tile_posi[1]):
        child = nodes.newNodes(
            minimum.mats,
            minimum.empty_tile_pos,
            new_tile_posi,
            minimum.levels + 1, # Use levels from minimum node
            minimum,
            final
        )
        pq.push(child)

initial = [[1, 2, 3], [5, 6, 0], [7, 8, 4]] #Ban đầu
empty_tile_posi = [1, 2]
final = [[1, 2, 3], [5, 8, 6], [0, 7, 4]] #Kết quả muốn trả về
solve(initial, empty_tile_posi, final)

1  2  3  
5  6  0  
7  8  4  

1  2  3  
5  0  6  
7  8  4  

1  2  3  
5  8  6  
7  0  4  

1  2  3  
5  8  6  
0  7  4  



#BTVN: Thuật toán AKT vào trong 15-puzzle

In [8]:
import copy
from heapq import heappush, heappop

# 15-puzzle (n=4)
n = 4

# 4 possible directions: down, left, up, right
rows = [1, 0, -1, 0]
cols = [0, -1, 0, 1]

class priorityQueue:
    def __init__(self):
        self.heap = []

    def push(self, key):
        heappush(self.heap, key)

    def pop(self):
        return heappop(self.heap)

    def empty(self):
        return not self.heap

class nodes:
    def __init__(self, parent, mats, empty_tile_pos, costs, levels):
        self.parent = parent
        self.mats = mats
        self.empty_tile_pos = empty_tile_pos
        self.costs = costs
        self.levels = levels

    def __lt__(self, nxt):
        return (self.costs + self.levels) < (nxt.costs + nxt.levels)

    def calculateCosts(mats, final) -> int:
        # Manhattan Distance Heuristic
        dist = 0
        goal_positions = {}
        for i in range(n):
            for j in range(n):
                goal_positions[final[i][j]] = (i, j)

        for i in range(n):
            for j in range(n):
                val = mats[i][j]
                if val != 0:
                    target_x, target_y = goal_positions[val]
                    dist += abs(i - target_x) + abs(j - target_y)
        return dist

    def newNodes(mats, empty_tile_posi, new_empty_tile_posi, level, parent, final):
        new_mats = copy.deepcopy(mats)
        x1, y1 = empty_tile_posi
        x2, y2 = new_empty_tile_posi

        # Swap tiles
        new_mats[x1][y1], new_mats[x2][y2] = new_mats[x2][y2], new_mats[x1][y1]

        # Calculate costs (h) for the new state
        costs = nodes.calculateCosts(new_mats, final)

        return nodes(parent, new_mats, new_empty_tile_posi, costs, level)


    def printMatrix(mats):
        for row in mats:
            print(" ".join(f"{val:2d}" for val in row))


    def isSafe(x, y):
        return 0 <= x < n and 0 <= y < n


    def printPath(root):
        if root is None:
            return
        nodes.printPath(root.parent)
        nodes.printMatrix(root.mats)
        print("----")

def solve(initial, empty_tile_posi, final):
    pq = priorityQueue()
    costs = nodes.calculateCosts(initial, final)
    root = nodes(None, initial, empty_tile_posi, costs, 0)
    pq.push(root)

    visited = set()

    while not pq.empty():
        minimum = pq.pop()

        state_tuple = tuple(map(tuple, minimum.mats))
        if state_tuple in visited:
            continue
        visited.add(state_tuple)

        if minimum.costs == 0:
            print(f"Solved in {minimum.levels} steps:")
            nodes.printPath(minimum)
            return

        for i in range(4):
            new_pos = [minimum.empty_tile_pos[0] + rows[i], minimum.empty_tile_pos[1] + cols[i]]
            if nodes.isSafe(new_pos[0], new_pos[1]):
                child = nodes.newNodes(minimum.mats, minimum.empty_tile_pos, new_pos, minimum.levels + 1, minimum, final)
                pq.push(child)

initial = [[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 0, 11], [13, 14, 15, 12]]
empty_tile_posi = [2, 2]
final = [[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 0]]

solve(initial, empty_tile_posi, final)

Solved in 2 steps:
 1  2  3  4
 5  6  7  8
 9 10  0 11
13 14 15 12
----
 1  2  3  4
 5  6  7  8
 9 10 11  0
13 14 15 12
----
 1  2  3  4
 5  6  7  8
 9 10 11 12
13 14 15  0
----
